In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import KFold
from sklearn.utils import resample

from scipy.stats import pearsonr

# Configuration

In [ ]:
target_in = 'FValue'  # 'FValue' (phi_tau) or 'W1' (Delta_tau)

train_path = './training_data/bias_train_e3.csv'
test_path = './training_data/bias_val_e3.csv'

k_folds = 10
boots_n = 1000

# Load Data

In [ ]:
data_train = pd.read_csv(train_path)
data_val = pd.read_csv(test_path)

In [ ]:
def model_fit(y_true, y_pred, verbose=False):
    pr = pearsonr(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    var = np.var(y_true)

    if verbose:
        print(f"R^2: {r2}")
        print(f"Pearson R: {pr[0]}")
        print(f"MSE: {mse}")
        print(f"RMSE: {mse**0.5}")
        print(f"NMSE: {mse/var}")
        print(f"NRMSE: {(mse**0.5)/var}")

    return r2, pr[0], mse, mse**0.5, mse/var, (mse**0.5)/var

# Prepare Feature Sets (Linear, Quadratic, Joint, All)

In [ ]:
feats_lin = ['x1', 'x2', 'x3', 'x4', 'x5', 'x6']
feats_quad = []
feats_joint = []

for feat in feats_lin:
    feats_quad.append(feat)
    feats_joint.append(feat)

for i, feat in enumerate(feats_lin):
    feats_quad.append(feat+feat)
    j = i+1
    while j < len(feats_lin):
        feats_joint.append(feat+feats_lin[j])
        j += 1

def build_features(data):
    df = pd.DataFrame()
    df['x1'] = data['gamma1']
    df['x2'] = data['lambda1']
    df['x3'] = data['delta']
    df['x4'] = data['epsilon']
    df['x5'] = data['NRow']
    df['x6'] = data['NCol']

    # Joint terms
    for i, fi in enumerate(feats_lin):
        for j in range(i+1, len(feats_lin)):
            df[fi+feats_lin[j]] = df[fi] * df[feats_lin[j]]

    # Squared terms
    for feat in feats_lin:
        df[feat+feat] = df[feat]**2

    return df

data_train_all = build_features(data_train)
data_val_all = build_features(data_val)

data_train_lin = data_train_all[feats_lin]
data_train_quad = data_train_all[feats_quad]
data_train_joint = data_train_all[feats_joint]

data_val_lin = data_val_all[feats_lin]
data_val_quad = data_val_all[feats_quad]
data_val_joint = data_val_all[feats_joint]

y_train = np.array(data_train[target_in])
y_val = np.array(data_val[target_in])

# 10-Fold Cross Validation for Model Selection

In [ ]:
kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)

variant_names = ['Linear', 'Quadratic', 'Joint', 'All']
train_sets = [np.array(data_train_lin), np.array(data_train_quad),
              np.array(data_train_joint), np.array(data_train_all)]

best_models = [None] * 4
best_nmses = [float('inf')] * 4
error_metrics_all = np.zeros((k_folds, 4))

for fold_idx, (train_index, test_index) in enumerate(kf.split(train_sets[0])):

    y_train_split = y_train[train_index]
    y_test_split = y_train[test_index]

    for v in range(4):
        X_tr = train_sets[v][train_index]
        X_te = train_sets[v][test_index]

        reg = LinearRegression()
        reg.fit(X_tr, y_train_split)

        y_pred = reg.predict(X_te)
        nmse = model_fit(y_test_split, y_pred)[4]

        error_metrics_all[fold_idx, v] = nmse

        if nmse < best_nmses[v]:
            best_nmses[v] = nmse
            best_models[v] = reg

print('Cross-validated NMSE (mean over folds):')
for v in range(4):
    print(f'  {variant_names[v]}: {np.mean(error_metrics_all[:, v]):.4f}')

# Bootstrapped Test and Train Performance

In [ ]:
val_sets = [np.array(data_val_lin), np.array(data_val_quad),
            np.array(data_val_joint), np.array(data_val_all)]

metrics_boot = np.zeros((boots_n, 4))
metrics_insample = np.zeros((boots_n, 4))

for i in range(boots_n):
    for v in range(4):
        X_boot, y_boot = resample(val_sets[v], y_val, n_samples=1000, random_state=42+i)
        y_pred = best_models[v].predict(X_boot)
        metrics_boot[i, v] = model_fit(y_boot, y_pred)[4]

        X_boot_tr, y_boot_tr = resample(train_sets[v], y_train, n_samples=1000, random_state=42+i)
        y_pred_tr = best_models[v].predict(X_boot_tr)
        metrics_insample[i, v] = model_fit(y_boot_tr, y_pred_tr)[4]

print(f'Results for target: {target_in}')
print('\nTest NMSE (mean +/- std):')
for v in range(4):
    print(f'  {variant_names[v]}: {np.mean(metrics_boot[:, v]):.3f} +/- {np.std(metrics_boot[:, v]):.3f}')
print('\nTrain NMSE (mean +/- std):')
for v in range(4):
    print(f'  {variant_names[v]}: {np.mean(metrics_insample[:, v]):.3f} +/- {np.std(metrics_insample[:, v]):.3f}')

In [ ]:
save_data = np.stack([metrics_boot, metrics_insample])  # shape (2, boots_n, 4)
np.save(f'./outputs/{target_in}_bootstrap_results.npy', save_data)
np.save(f'./outputs/{target_in}_kfold_results.npy', error_metrics_all)